# graphArt_LM: Phase 2, LoRA fine-tuning on a free Kaggle GPU

Fine-tunes `Qwen2.5-Coder-1.5B-Instruct` on the 1280 training samples, then scores the
tuned adapter **and** the untuned base on the same 40 stratified test samples as every
Phase 1 run, with the `zero_shot` prompt. The test samples are never used in training.

**Before running**, in the right-hand panel under *Session options*:
- **Accelerator:** GPU T4 x2 (training uses one of them)
- **Internet:** on (to install the libraries and download the model)

Then *Save Version → Save & Run All (Commit)*. A 20-step smoke run comes first; if the loss
is not falling (or overflows in fp16, after one retry in fp32) the session stops there.

**When it finishes**, download `graphart_finetune.zip` from the *Output* tab and extract it
into a new folder; it holds `results/` (both runs) and the adapter under `lora/`.

Generated by `scripts/make_kaggle_notebook.py --finetune`. Edit that script, not this file.

In [ ]:
# BASE_MODEL   the Hugging Face model to tune -- the same weights as Ollama's
#              qwen2.5-coder:1.5b, unquantised.
# ADAPTER      the adapter's name; its results land in results/raw/hf__<ADAPTER>__zero_shot/.
# EPOCHS       passes over the 1280 train samples. The val split picks the best checkpoint.
# SMOKE_STEPS  a short first run that must show the loss falling before the real one starts.
# TIME_BUDGET  the whole session; training stops early enough to leave an hour for scoring.
# TRAIN_ARGS / EVAL_ARGS  extra flags for train_lora.py / run_prompting.py (normally empty).
# VALIDATE_ONLY True: only check the code arrived intact (works on a CPU session).
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"; ADAPTER = "qwen2.5-coder-1.5b-graphart-lora"; EPOCHS = 2.0; SMOKE_STEPS = 20; TIME_BUDGET_HOURS = 11.0; TRAIN_ARGS = []; EVAL_ARGS = []; VALIDATE_ONLY = False

## 1. Unpack the code, prompts and samples

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

START = time.time()
WORK = Path(os.environ.get("GRAPHART_WORK", "/kaggle/working"))
REPO = WORK / "graphArt_LM"
BUNDLE_SHA256 = "c51109e3cef81ed1920d965f57b406ef30144a04ecf4652f38fe9c73f18c81aa"
BUNDLE = (
    "..."
    "c/1z/XP9c/0T9+f/AbDiL4AAICYA"
)
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256, "bundle damaged -- regenerate the notebook"
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tar:
    tar.extractall(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()))
if VALIDATE_ONLY:
    raise SystemExit("bundle intact -- VALIDATE_ONLY is set, so stopping here")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## 2. Install the pinned libraries

In [ ]:
from importlib.metadata import PackageNotFoundError, version

PINS = {'transformers': '5.20.2', 'peft': '0.21.2', 'accelerate': '1.15.0'}

def pinned(package, wanted):
    try:
        return version(package) == wanted
    except PackageNotFoundError:
        return False

missing = [f"{p}=={v}" for p, v in PINS.items() if not pinned(p, v)]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# peft probes two optional quantisation libraries when it builds LoRA layers, and
# raises -- rather than skipping them -- if they are installed but older than it
# supports. Kaggle ships torchao 0.10 (peft wants >= 0.16), which killed a run. We
# use neither, so remove them if present.
for pkg in ("torchao", "gptqmodel"):
    try:
        print(f"removing {pkg} {version(pkg)} (peft rejects old versions; not used here)", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg], check=True)
    except PackageNotFoundError:
        pass

subprocess.run([sys.executable, "-c", "import torch, transformers, peft; "
                "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
                "'| transformers', transformers.__version__, '| peft', peft.__version__)"])

## 3. Download the base model

In [ ]:
# Its own step with a time limit: a stalled download inside training would otherwise
# sit until the session cap. ~3 GB; a few minutes on Kaggle's network.
FETCH = ("from huggingface_hub import snapshot_download; "
         f"print('downloaded to', snapshot_download({BASE_MODEL!r}))")
for attempt in (1, 2):
    try:
        subprocess.run([sys.executable, "-c", FETCH], check=True, timeout=1800)
        break
    except (subprocess.TimeoutExpired, subprocess.CalledProcessError) as exc:
        print(f"download attempt {attempt} failed: {exc}", flush=True)
else:
    raise SystemExit("could not download the base model -- stopping before any training")

## 4. Smoke run: is it learning?

In [ ]:
# One GPU is plenty for a 1.5B with LoRA, and avoids DataParallel's quirks. Offline from
# here on: the model is on disk, and nothing later should wait on the network.
GPU_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES="0", HF_HUB_OFFLINE="1")

def train(*extra):
    return subprocess.run([sys.executable, "scripts/train_lora.py", "--base", BASE_MODEL,
                           *TRAIN_ARGS, *extra], env=GPU_ENV).returncode

# Every train/val example must fit; checked in seconds, before the GPU does anything.
if train("--out", "lora/smoke", "--check-only") != 0:
    raise SystemExit("the data check failed -- see the traceback above")

# train_lora.py exits 3 when the loss is not falling and 4 when it is NaN/inf. Only
# those two can be an fp16 problem (Qwen2.5 can overflow on a T4), so only they get a
# retry in fp32 (~2x slower). Any other exit code is a crash: stop and read it.
RETRYABLE = (3, 4)
code = train("--out", "lora/smoke", "--smoke-steps", str(SMOKE_STEPS))
if code in RETRYABLE:
    print(f"smoke run failed in fp16 (exit {code}) -- retrying in fp32", flush=True)
    code = train("--out", "lora/smoke", "--smoke-steps", str(SMOKE_STEPS), "--fp32")
    if code == 0:
        TRAIN_ARGS = [*TRAIN_ARGS, "--fp32"]
if code == 3:
    raise SystemExit("the smoke run did not learn, in fp16 or fp32 -- stopping before the full run")
if code != 0:
    raise SystemExit(f"the smoke run crashed (exit {code}) -- see the traceback above")

## 5. Train

In [ ]:
import zipfile

def package():
    # The results and the adapter, without the optimizer-state checkpoints.
    with zipfile.ZipFile(WORK / "graphart_finetune.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for root in (REPO / "results", REPO / "lora" / ADAPTER):
            for p in root.rglob("*") if root.exists() else []:
                if p.is_file() and "checkpoints" not in p.parts:
                    z.write(p, p.relative_to(REPO))

hours_left = TIME_BUDGET_HOURS - (time.time() - START) / 3600 - 1.0   # an hour kept for scoring
if train("--out", f"lora/{ADAPTER}", "--epochs", str(EPOCHS),
         "--time-budget-hours", f"{hours_left:.2f}") != 0:
    raise SystemExit("training failed -- see the log above")
print((REPO / "lora" / ADAPTER / "train_summary.json").read_text())
package()

## 6. Score the tuned adapter and the untuned base

In [ ]:
for model in [str(REPO / "lora" / ADAPTER), BASE_MODEL]:
    subprocess.run([sys.executable, "-m", "pipeline.run_prompting", "--backend", "hf",
                    "--model", model, "--prompt", "zero_shot", "--stratify", "10", *EVAL_ARGS],
                   env=GPU_ENV)
    package()
for meta in sorted((REPO / "results").glob("*.meta.json")):
    m = json.loads(meta.read_text())
    print(f"{meta.stem:60s} n={m['n']} failed={m['failed']}")
print(f"done in {(time.time() - START) / 3600:.1f} h -- download graphart_finetune.zip")